# Sync Registry to AgentCore Gateway

The Sync Lambda is designed to bridge the Module 3 Registry (the tool catalog) and the AgentCore Gateway (the governed runtime). In production, EventBridge triggers it every 5 minutes. In this notebook you invoke it manually and inspect the outcome.

**Important — what you should expect to see:**

You already created the three gateway targets manually in notebook 03 (`tg-workshop-flights-mcp`, `tg-workshop-hotels-mcp`, `tg-search-knowledge-base`). Your Registry entries for those tools use HTTPS Lambda Function URLs (`https://xxx.lambda-url.us-west-2.on.aws/`), not the `lambda://` scheme the Sync Lambda keys on. As a result, running the Sync Lambda here will produce:

- `created: 0` — nothing new to create
- `skipped: N` — every Registry server is either already represented (dedup) or uses a non-`lambda://` URL

This is the expected steady state, not a failure. The Sync Lambda becomes productive when a team registers a tool with a `lambda://` URL in the Registry — it then materializes a matching gateway target automatically.

**How the Sync Lambda works:**

1. Authenticates to the Registry API using M2M credentials from Secrets Manager
2. Fetches all registered servers via `GET /api/servers`
3. Lists existing gateway targets to avoid duplicates
4. For each server, checks the `proxy_pass_url`:
   - `lambda://...` creates a native Lambda target in the gateway
   - `http://` or `https://` servers are skipped (they use Path A via CloudFront/NGINX)
5. If `SYNC_FILTER_TAGS` is set, only tools with matching tags are synced
6. Creates new targets, skips existing ones, reports results

## Step 1: Invoke the Sync Lambda

We invoke the Lambda synchronously using the AWS SDK. The response payload contains counts of created, skipped, and errored targets.

In [ ]:
import boto3, json

region = boto3.session.Session().region_name or "us-west-2"
lambda_client = boto3.client("lambda", region_name=region)

SYNC_FN_NAME = "agentcore-gateway-sync"

print(f"Invoking {SYNC_FN_NAME}...")
print("=" * 60)

response = lambda_client.invoke(
    FunctionName=SYNC_FN_NAME,
    InvocationType="RequestResponse",  # Synchronous
    Payload=json.dumps({"source": "notebook", "manual": True}),
)

# Read the response payload
payload = json.loads(response["Payload"].read().decode("utf-8"))
status_code = response.get("StatusCode", "?")

print(f"Status Code: {status_code}")
print(f"Response:")
print(json.dumps(payload, indent=2))

# Check for Lambda errors
if "FunctionError" in response:
    print(f"\nLambda error type: {response['FunctionError']}")
    print("Check CloudWatch logs for agentcore-gateway-sync for details.")
else:
    synced = payload.get("synced", 0)
    created = payload.get("created", 0)
    skipped = payload.get("skipped", 0)
    errors = payload.get("errors", 0)
    print(f"\nSync Results:")
    print(f"  Servers processed: {synced}")
    print(f"  Targets created:   {created}")
    print(f"  Targets skipped:   {skipped} (already exist)")
    print(f"  Errors:            {errors}")
    if errors > 0:
        raise RuntimeError(
            f"Sync Lambda reported {errors} error(s). "
            f"Check /aws/lambda/{SYNC_FN_NAME} in CloudWatch for details."
        )
if "FunctionError" in response:
    raise RuntimeError(
        f"Sync Lambda FunctionError={response['FunctionError']}: {json.dumps(payload)[:400]}"
    )

## Step 2: List Gateway Targets

Now let's query the AgentCore Gateway directly to see what targets were created. Each target corresponds to a tool in the Registry.

In [ ]:
# Look up the gateway ID
agentcore = boto3.client("bedrock-agentcore-control", region_name=region)
gateways = agentcore.list_gateways().get("items", [])
gw = [g for g in gateways if g["name"] == "tools-gateway"]
if not gw:
    raise RuntimeError("Gateway 'tools-gateway' not found!")

GATEWAY_ID = gw[0]["gatewayId"]
print(f"Gateway: {gw[0]['name']} (ID: {GATEWAY_ID}, Status: {gw[0]['status']})")
print()

# List all targets
targets = agentcore.list_gateway_targets(gatewayIdentifier=GATEWAY_ID).get("items", [])

print(f"Gateway Targets ({len(targets)} total):")
print("=" * 80)
print(f"  {'Name':35s}  {'Status':12s}  {'Target ID'}")
print("-" * 80)
for t in targets:
    name = t.get("name", "?")
    status = t.get("status", "?")
    target_id = t.get("targetId", "?")
    print(f"  {name:35s}  {status:12s}  {target_id}")

## Step 3: Verify Expected Targets

Let's confirm the three targets you created manually in notebook 03 are present in the Gateway. The Sync Lambda did not create them — it skipped them because their Registry `proxy_pass_url` uses `https://` (Lambda Function URL), which Path A already handles via CloudFront/NGINX. The Sync Lambda reserves target creation for `lambda://`-scheme servers that only Path B can serve.

- `workshop-flights-mcp` -- Present (created manually in notebook 03)
- `workshop-hotels-mcp` -- Present (created manually in notebook 03)
- `search-knowledge-base` -- Present (created manually in notebook 03)
- `currenttime-server` -- Skipped (Docker backend, `http://`)
- `realserverfaketools` -- Skipped (Docker backend, `http://`)

In [ ]:
# Expected targets (the "tg-" prefix is added by the Sync Lambda)
# All three are Lambda tools created in step 3 with lambda:// URLs.
# The Sync Lambda only creates gateway targets for lambda:// entries;
# Docker servers (currenttime-server, realserverfaketools) are skipped.
expected_new_tools = [
    "tg-workshop-flights-mcp",       # Lambda MCP tool (created in step 3)
    "tg-workshop-hotels-mcp",        # Lambda MCP tool (created in step 3)
    "tg-search-knowledge-base",      # Lambda MCP tool (created in step 3)
]

target_names = {t.get("name", "") for t in targets}

print("Verification of gateway targets:")
print("-" * 50)
all_ok = True
for expected in expected_new_tools:
    found = expected in target_names
    status = "FOUND" if found else "MISSING"
    marker = "  " if found else "  "
    print(f"  {marker} {expected:35s}  [{status}]")
    if not found:
        all_ok = False

print()
if all_ok:
    print("All expected targets are present in the AgentCore Gateway!")
else:
    print("Some targets are missing. This could mean:")
    print("  - The Sync Lambda encountered errors (check CloudWatch logs)")
    print("  - The tools were not registered in notebook 03")
    print("  - Try running the Sync Lambda again (re-run Step 1)")

print(f"\nTotal targets in gateway: {len(targets)}")
print("Note: Docker servers (currenttime-server, realserverfaketools) are skipped by the Sync Lambda.")

## Step 4: Tag-Based Tool Selection

In production, the Registry may contain dozens of tools — but you don't want ALL of them in the gateway. The `SYNC_FILTER_TAGS` environment variable lets the platform team **choose** which tools become gateway targets.

**How it works:**
- Each tool in the Registry can have `tags` (e.g., `["agentcore-target", "production"]`)
- Set `SYNC_FILTER_TAGS` on the Sync Lambda to a comma-separated list of tags
- Only tools with at least one matching tag are synced to the gateway
- Empty value (default) = sync all tools (current behavior)

This is the "choose which tools" mechanism — platform teams tag tools for gateway inclusion, and the Sync Lambda respects those tags.

In [ ]:
# Check the current SYNC_FILTER_TAGS setting on the Sync Lambda
config = lambda_client.get_function_configuration(FunctionName=SYNC_FN_NAME)
env_vars = config.get("Environment", {}).get("Variables", {})
current_filter = env_vars.get("SYNC_FILTER_TAGS", "")

print(f"Current SYNC_FILTER_TAGS: {current_filter!r}")
if not current_filter:
    print("  (empty — all tools are synced regardless of tags)")
else:
    print(f"  Only tools tagged with: {current_filter.split(',')}")

In [ ]:
# Enable tag-based filtering: only sync tools tagged "agentcore-target"
# This uses read-merge-update to preserve all other env vars.

FILTER_TAG = "agentcore-target"

# Read current env vars
config = lambda_client.get_function_configuration(FunctionName=SYNC_FN_NAME)
env_vars = config.get("Environment", {}).get("Variables", {})

# Merge in the filter tag
env_vars["SYNC_FILTER_TAGS"] = FILTER_TAG

print(f"Setting SYNC_FILTER_TAGS = {FILTER_TAG!r}")
print("Updating Lambda configuration...")

lambda_client.update_function_configuration(
    FunctionName=SYNC_FN_NAME,
    Environment={"Variables": env_vars},
)

# Use the AWS waiter — correctly handles Failed/InProgress states
import time
lambda_client.get_waiter("function_updated_v2").wait(
    FunctionName=SYNC_FN_NAME, WaiterConfig={"Delay": 2, "MaxAttempts": 30}
)
final = lambda_client.get_function_configuration(FunctionName=SYNC_FN_NAME)
status = final["LastUpdateStatus"]
if status != "Successful":
    reason = final.get("LastUpdateStatusReason", "")
    raise RuntimeError(
        f"Lambda {SYNC_FN_NAME} update finished in state {status}: {reason}"
    )
print(f"Update status: {status}")
print(f"\nNow only tools tagged '{FILTER_TAG}' will be synced to the gateway.")

In [ ]:
# Re-run sync with tag filtering enabled
# Tools WITHOUT the "agentcore-target" tag will be skipped.

print(f"Invoking {SYNC_FN_NAME} with tag filter active...")
print("=" * 60)

response = lambda_client.invoke(
    FunctionName=SYNC_FN_NAME,
    InvocationType="RequestResponse",
    Payload=json.dumps({"source": "notebook", "manual": True}),
)

payload = json.loads(response["Payload"].read().decode("utf-8"))

if "FunctionError" in response:
    raise RuntimeError(
        f"Sync Lambda FunctionError={response['FunctionError']}: {json.dumps(payload)[:400]}"
    )
else:
    created = payload.get("created", 0)
    skipped = payload.get("skipped", 0)
    filtered = payload.get("filtered", 0)
    errors = payload.get("errors", 0)
    if errors > 0:
        raise RuntimeError(
            f"Sync Lambda reported {errors} error(s) under tag filter. "
            f"Check /aws/lambda/{SYNC_FN_NAME} for details."
        )
    print(f"Sync Results (with SYNC_FILTER_TAGS={FILTER_TAG!r}):")
    print(f"  Targets created:   {created}")
    print(f"  Targets skipped:   {skipped} (already exist)")
    print(f"  Tools filtered:    {filtered} (tag not matched)")
    print(f"  Errors:            {errors}")
    print()
    if filtered > 0:
        print(f"  {filtered} tool(s) were NOT synced because they lack the '{FILTER_TAG}' tag.")
        print("  This is the platform team's selection mechanism in action.")

In [ ]:
# Reset: clear the filter so all tools sync again (for the remaining notebooks)
config = lambda_client.get_function_configuration(FunctionName=SYNC_FN_NAME)
env_vars = config.get("Environment", {}).get("Variables", {})
env_vars["SYNC_FILTER_TAGS"] = ""

lambda_client.update_function_configuration(
    FunctionName=SYNC_FN_NAME,
    Environment={"Variables": env_vars},
)

lambda_client.get_waiter("function_updated_v2").wait(
    FunctionName=SYNC_FN_NAME, WaiterConfig={"Delay": 2, "MaxAttempts": 30}
)
final = lambda_client.get_function_configuration(FunctionName=SYNC_FN_NAME)
if final["LastUpdateStatus"] != "Successful":
    raise RuntimeError(
        f"Failed to clear SYNC_FILTER_TAGS on {SYNC_FN_NAME}: "
        f"{final.get('LastUpdateStatusReason', '')}"
    )

print(f"SYNC_FILTER_TAGS reset to empty (sync all tools).")
print("This ensures notebooks 05-07 see all targets.")

## Summary

You have completed the sync step. Here is what you built:

```
Registry (Module 3)                    AgentCore Gateway (Module 4)
+-----------------------------+        +------------------------------+
| workshop-flights-mcp        | -----> | tg-workshop-flights-mcp      |
|   (lambda:// URL)           |        |   (native Lambda target)     |
| workshop-hotels-mcp         | -----> | tg-workshop-hotels-mcp       |
|   (lambda:// URL)           |        |   (native Lambda target)     |
| search-knowledge-base       | -----> | tg-search-knowledge-base     |
|   (lambda:// URL)           |        |   (native Lambda target)     |
+-----------------------------+        +------------------------------+
| currenttime-server (http://)  |  (Skipped -- Docker tools use NGINX)
| realserverfaketools (http://) |  (Skipped -- Docker tools use NGINX)
+-----------------------------+

         Path A (NGINX)                       Path B (Governed)
    - Direct proxy via CloudFront         - Cognito JWT auth
    - Docker + HTTP tools                 - Audit interceptor
    - No audit trail                      - Guardrails interceptor
                                          - Lambda-native targets
```

**Key takeaways:**
- The Registry is the single source of truth for all tools, regardless of which path serves them.
- The Sync Lambda keeps the AgentCore Gateway's target list synchronized with the Registry catalog.
- Only `lambda://` tools become gateway targets -- Docker servers continue to use Path A (NGINX).
- Path B extends the platform with Lambda-native tools that Path A cannot serve natively.
- Every request through Path B is audit-logged; every response can be screened by Bedrock Guardrails.
- Both paths coexist -- agents choose the path that matches their governance requirements.